# AethyxLM - Production Kaggle Training (T4 GPU x2)

**Architecture:** Decoder-only GPT (14M params, 8L, 256D, 8H, 128ctx, 32k vocab)
**Dataset:** TinyStories (auto-download from Hugging Face)
**Storage:** GitHub = code, `/kaggle/working` = persistent checkpoints/logs/data/configs

---

In [62]:
# ============================================================
# CELL 1: SETUP PROJECT (Kaggle)
# ============================================================
import os, sys, subprocess, shutil, json, time, glob, signal
from pathlib import Path

# Kaggle working directory
WORK_DIR = '/kaggle/working'
os.chdir(WORK_DIR)

# Project root
LOCAL_ROOT = os.path.join(WORK_DIR, 'AethyxLM')

# Persistent directories on Kaggle working dir
CKPT_DIR = os.path.join(WORK_DIR, 'checkpoints')
LOGS_DIR = os.path.join(WORK_DIR, 'logs')
TOK_DIR = os.path.join(WORK_DIR, 'tokenizer')
DATA_DIR = os.path.join(WORK_DIR, 'data')
CONFIG_DIR = os.path.join(WORK_DIR, 'configs')

for d in [CKPT_DIR, LOGS_DIR, TOK_DIR, DATA_DIR, CONFIG_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'[OK] Working dir: {WORK_DIR}')
print(f'[OK] Checkpoints: {CKPT_DIR}')
print(f'[OK] Logs: {LOGS_DIR}')
print(f'[OK] Configs: {CONFIG_DIR}')

# Clone or update Git repository
REPO_URL = 'https://github.com/aethyx-ai/AethyxLM.git'

if os.path.exists(os.path.join(LOCAL_ROOT, '.git')):
    print('Updating existing repository...')
    subprocess.run(['git', '-C', LOCAL_ROOT, 'pull'], check=False)
else:
    print('Cloning repository...')
    subprocess.run(['git', 'clone', REPO_URL, LOCAL_ROOT], check=True)

# Fix nested directory if created during clone
nested = os.path.join(LOCAL_ROOT, 'AethyxLM')
if os.path.exists(nested) and os.path.isdir(nested):
    for item in os.listdir(nested):
        src = os.path.join(nested, item)
        dst = os.path.join(LOCAL_ROOT, item)
        if os.path.exists(dst):
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            else:
                os.remove(dst)
        shutil.move(src, LOCAL_ROOT)
    try:
        os.rmdir(nested)
    except Exception:
        pass

os.chdir(LOCAL_ROOT)
if LOCAL_ROOT not in sys.path:
    sys.path.insert(0, LOCAL_ROOT)

print(f'[OK] Project root: {LOCAL_ROOT}')

# Install required dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', 'tokenizers', 'datasets', 'tensorboard', '-q'], check=True)
print('[OK] Dependencies installed.')

# ============================================================


[OK] Working dir: /kaggle/working
[OK] Checkpoints: /kaggle/working/checkpoints
[OK] Logs: /kaggle/working/logs
[OK] Configs: /kaggle/working/configs
Updating existing repository...


From https://github.com/aethyx-ai/AethyxLM
   7611fcf..92ee0db  main       -> origin/main


Updating 7611fcf..92ee0db
Fast-forward
 AethyxLM/kaggle_train_production.ipynb | 125 ++++++++++++++++++++++++++++-----
 AethyxLM/train.py                      |   4 ++
 2 files changed, 110 insertions(+), 19 deletions(-)
[OK] Project root: /kaggle/working/AethyxLM
[OK] Dependencies installed.


In [63]:
# CELL 2: VERIFY CUDA ACCELERATOR
# ============================================================
import os, torch

# Make sure the Jupyter kernel can see the GPUs that Kaggle attached.
# Use both GPUs (T4 x2) for DDP training
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not detected! Ensure the notebook is running with a GPU accelerator (Settings -> Accelerator -> GPU)')
else:
    device_count = torch.cuda.device_count()
    print(f'Number of GPUs: {device_count}')
    for i in range(device_count):
        device_name = torch.cuda.get_device_name(i)
        vram_gb = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'GPU {i}: {device_name} ({vram_gb:.1f} GB VRAM)')
    device = "cuda"

# Known GPU models we expect on Kaggle – informational only
known_gpus = ['T4', 'L4', 'A100', 'V100', 'P100', 'K80', 'A10G']
if any(g in device_name for g in known_gpus):
    print(f'GPU {device_name} recognized - proceeding with training.')
else:
    print(f'Notice: GPU "{device_name}" detected. Proceeding with training anyway.')


PyTorch version: 2.10.0+cu128
CUDA available: True
Number of GPUs: 2
GPU 0: Tesla T4 (15.6 GB VRAM)
GPU 1: Tesla T4 (15.6 GB VRAM)
GPU Tesla T4 recognized - proceeding with training.


In [64]:
# ============================================================
# CELL 3: PREPARE DATA (TinyStories from Hugging Face)
# ============================================================
import random, os, shutil
from pathlib import Path
from datasets import load_dataset

print('Loading TinyStories dataset from Hugging Face...')
ds = load_dataset('roneneldan/TinyStories', split='train')

NUM_STORIES = 100_000  # Start with 100k stories; increase after first successful run
texts = [item['text'] for item in ds.select(range(min(NUM_STORIES, len(ds))))]

random.seed(42)
random.shuffle(texts)
split = int(0.95 * len(texts))
train_texts = texts[:split]
val_texts = texts[split:]

os.makedirs('data', exist_ok=True)
os.makedirs('tokenizer/data', exist_ok=True)

train_content = '\n\n'.join(train_texts)
val_content = '\n\n'.join(val_texts)

with open('data/train.txt', 'w', encoding='utf-8') as f:
    f.write(train_content)
with open('data/val.txt', 'w', encoding='utf-8') as f:
    f.write(val_content)

# Save corpus for tokenizer trainer
with open('tokenizer/data/corpus.txt', 'w', encoding='utf-8') as f:
    f.write(train_content)

print(f'Train: {len(train_texts)} stories saved to data/train.txt')
print(f'Val: {len(val_texts)} stories saved to data/val.txt')

# Persistent storage backup
shutil.copy('data/train.txt', os.path.join(DATA_DIR, 'train.txt'))
shutil.copy('data/val.txt', os.path.join(DATA_DIR, 'val.txt'))
shutil.copy('tokenizer/data/corpus.txt', os.path.join(TOK_DIR, 'corpus.txt'))

print('[OK] Dataset prepared and backed up to persistent storage.')

Loading TinyStories dataset from Hugging Face...
Train: 95000 stories saved to data/train.txt
Val: 5000 stories saved to data/val.txt
[OK] Dataset prepared and backed up to persistent storage.


In [65]:
# ============================================================
# CELL 4: TRAIN TOKENIZER (BPE, 32k vocab)
# ============================================================
import subprocess, sys, shutil, os

print('Training AethyxTokenizer via tokenizer.train_tokenizer...')
result = subprocess.run(
    [sys.executable, '-m', 'tokenizer.train_tokenizer'],
    cwd=LOCAL_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Tokenizer training failed!')

# Verify in Python
from tokenizer.tokenizer import AethyxTokenizer
tok = AethyxTokenizer()
print(f'[OK] Tokenizer loaded successfully. Vocab size: {tok.vocab_size}')

# Backup to persistent storage
if os.path.exists('tokenizer/tokenizer.json'):
    shutil.copy2('tokenizer/tokenizer.json', os.path.join(TOK_DIR, 'tokenizer.json'))
if os.path.exists('tokenizer/metadata.json'):
    shutil.copy2('tokenizer/metadata.json', os.path.join(TOK_DIR, 'metadata.json'))

print('[OK] Tokenizer backed up to persistent storage.')

Training AethyxTokenizer via tokenizer.train_tokenizer...



Training AethyxTokenizer...

Training complete!
Tokenizer saved to:
/kaggle/working/AethyxLM/tokenizer/tokenizer.json
Metadata saved to:
/kaggle/working/AethyxLM/tokenizer/metadata.json

[OK] Tokenizer loaded successfully. Vocab size: 20295
[OK] Tokenizer backed up to persistent storage.


In [66]:
# ============================================================
# CELL 5: LOAD & CREATE KAGGLE TRAINING CONFIG
# ============================================================
import json, os, shutil

base_config_path = 'configs/train_config.json'
if not os.path.exists(base_config_path):
    base_config_path = 'configs/train_config_kaggle.json'

with open(base_config_path, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

# Optimize hyper-parameters for Kaggle GPU session
cfg['training'].update({
    'learning_rate': 3e-4,
    'weight_decay': 0.1,
    'warmup_steps': 1000,
    'max_steps': 20000,
    'batch_size': 32,
    'grad_accum_steps': 1,
    'use_amp': True,
    'log_interval': 50,
    'eval_interval': 500,
    'save_interval': 1000,
    'generate_interval': 500
})
cfg['data'].update({
    'batch_size': 32,
    'num_workers': 2,
    'train_file': 'data/train.txt',
    'val_file': 'data/val.txt'
})

kaggle_config_path = 'configs/train_config_kaggle.json'
with open(kaggle_config_path, 'w', encoding='utf-8') as f:
    json.dump(cfg, f, indent=2)

shutil.copy2(kaggle_config_path, os.path.join(CONFIG_DIR, 'train_config_kaggle.json'))

print('[OK] Kaggle config written to configs/train_config_kaggle.json:')
print(json.dumps(cfg, indent=2))

[OK] Kaggle config written to configs/train_config_kaggle.json:
{
  "seed": 42,
  "model": {
    "vocab_size": 32000,
    "context_length": 128,
    "embed_dim": 256,
    "num_heads": 8,
    "num_layers": 8,
    "ffn_dim": 1024,
    "dropout": 0.1,
    "use_bias": true,
    "layer_norm_eps": 1e-05
  },
  "training": {
    "learning_rate": 0.0003,
    "weight_decay": 0.1,
    "betas": [
      0.9,
      0.95
    ],
    "eps": 1e-08,
    "grad_clip": 1.0,
    "warmup_steps": 1000,
    "max_steps": 20000,
    "min_lr_ratio": 0.1,
    "grad_accum_steps": 1,
    "use_amp": true,
    "batch_size": 32,
    "num_workers": 2,
    "eval_interval": 500,
    "save_interval": 1000,
    "log_interval": 50,
    "generate_interval": 500
  },
  "data": {
    "train_file": "data/train.txt",
    "val_file": "data/val.txt",
    "context_length": 128,
    "batch_size": 32,
    "num_workers": 2,
    "shuffle": true
  },
  "checkpoint": {
    "checkpoint_dir": "checkpoints",
    "log_interval": 10,
    "eval

In [67]:
# ============================================================
# CELL 6: AUTO-RESUME CHECKPOINT DETECTION
# ============================================================
import os, glob

def find_latest_checkpoint():
    """Find latest valid checkpoint in persistent storage or local."""
    candidates = []
    
    for base in [CKPT_DIR, 'checkpoints']:
        if os.path.exists(base):
            best = os.path.join(base, 'checkpoint_best.pt')
            latest = os.path.join(base, 'checkpoint_latest.pt')
            if os.path.exists(latest) and os.path.getsize(latest) > 1_000_000:
                candidates.append(latest)
            step_files = sorted(
                glob.glob(os.path.join(base, 'checkpoint_step_*.pt')),
                key=lambda x: int(x.split('_')[-1].split('.')[0]) if x.split('_')[-1].split('.')[0].isdigit() else 0
            )
            if step_files:
                valid_steps = [sf for sf in step_files if os.path.getsize(sf) > 1_000_000]
                if valid_steps:
                    candidates.append(valid_steps[-1])

    for c in candidates:
        if os.path.exists(c):
            return c
    return None

resume_path = find_latest_checkpoint()
if resume_path:
    print(f'[OK] Found existing checkpoint for auto-resume: {resume_path}')
    RESUME_ARGS = ['--resume', resume_path]
else:
    print('[OK] No previous checkpoint found. Starting fresh training run.')
    RESUME_ARGS = []

[OK] No previous checkpoint found. Starting fresh training run.


In [68]:
# ============================================================
# CELL 7: SYNC CHECKPOINTS + LOGS + CONFIG (LOCAL <-> PERSISTENT)
# ============================================================
import os, shutil

def sync_to_persistent():
    """Copy local checkpoints, logs, and config to persistent Kaggle working dir."""
    if os.path.exists('checkpoints'):
        for f in os.listdir('checkpoints'):
            if f.endswith('.pt'):
                try:
                    shutil.copy2(os.path.join('checkpoints', f), os.path.join(CKPT_DIR, f))
                except Exception as e:
                    print(f'  Checkpoint sync failed for {f}: {e}')

    if os.path.exists('logs'):
        for root, dirs, files in os.walk('logs'):
            rel_path = os.path.relpath(root, 'logs')
            target_dir = os.path.join(LOGS_DIR, rel_path) if rel_path != '.' else LOGS_DIR
            os.makedirs(target_dir, exist_ok=True)
            for f in files:
                try:
                    shutil.copy2(os.path.join(root, f), os.path.join(target_dir, f))
                except Exception as e:
                    print(f'  Log sync failed for {f}: {e}')

    kaggle_cfg = 'configs/train_config_kaggle.json'
    if os.path.exists(kaggle_cfg):
        try:
            shutil.copy2(kaggle_cfg, os.path.join(CONFIG_DIR, 'train_config_kaggle.json'))
        except Exception as e:
            print(f'  Config sync failed: {e}')

def sync_from_persistent():
    """Copy persistent checkpoints to local before training."""
    if not os.path.exists(CKPT_DIR):
        return
    os.makedirs('checkpoints', exist_ok=True)
    for f in os.listdir(CKPT_DIR):
        if f.endswith('.pt'):
            src = os.path.join(CKPT_DIR, f)
            dst = os.path.join('checkpoints', f)
            if not os.path.exists(dst) or os.path.getmtime(src) > os.path.getmtime(dst):
                try:
                    shutil.copy2(src, dst)
                    print(f'  Synced from persistent: {f}')
                except Exception as e:
                    print(f'  Sync failed for {f}: {e}')

# Perform initial sync from persistent storage
sync_from_persistent()
print('[OK] Sync manager ready.')

[OK] Sync manager ready.


In [69]:
# ============================================================
# CELL 8: TRAINING WRAPPER WITH REAL-TIME VSCODE STREAMING
# ============================================================
import os, sys, time, threading, subprocess, torch

# Force the process to see both GPUs that Kaggle attached.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0,1')

# Keep the Hugging Face datasets library offline - data is cached locally.

# Small GPU-memory clean-up before launching the trainer.
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('Starting training on GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('=' * 60)

# Define device locally (in case Cell 2 variable is out of scope)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Define RESUME_ARGS if not already defined by Cell 6
try:
    RESUME_ARGS
except NameError:
    RESUME_ARGS = []

# Build the command - use DDP if 2 GPUs available
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if num_gpus > 1:
    # DDP launch: torchrun spawns one process per GPU
    cmd = [
        sys.executable, '-m', 'torch.distributed.run',
        '--nproc_per_node=' + str(num_gpus),
        'train.py',
        '--config', 'configs/train_config_kaggle.json',
        '--ddp',
    ]
else:
    cmd = [
        sys.executable, 'train.py',
        '--config', 'configs/train_config_kaggle.json',
        '--device', device,
    ]

if RESUME_ARGS:
    cmd.extend(RESUME_ARGS)

print('Command:', ' '.join(cmd))
print('-' * 60)

stop_sync = False
sync_lock = threading.Lock()

def periodic_sync():
    while not stop_sync:
        time.sleep(300)  # Sync every 5 minutes
        if not stop_sync:
            with sync_lock:
                sync_to_persistent()

sync_thread = threading.Thread(target=periodic_sync, daemon=True)
sync_thread.start()

start_time = time.time()
try:
    process = subprocess.Popen(
        cmd,
        cwd=os.getcwd(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        print(line, end='', flush=True)

    process.wait()
    retcode = process.returncode
finally:
    stop_sync = True
    sync_thread.join(timeout=10)
    sync_to_persistent()
    elapsed = time.time() - start_time
    print('=' * 60)
    print(f'Training run completed in {elapsed/3600:.2f} hours.')

if retcode == 0:
    print('[OK] Training completed successfully!')
else:
    print(f'[FAIL] Process exited with code {retcode}')


Starting training on GPU: Tesla T4
Command: /usr/bin/python3 -m torch.distributed.run --nproc_per_node=2 train.py --config configs/train_config_kaggle.json --ddp
------------------------------------------------------------

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
Traceback (most recent call last):
  File "/kaggle/working/AethyxLM/train.py", line 22, in <module>
    from model.gpt import GPT
  File "/kaggle/working/AethyxLM/model/gpt.py", line 8, in <module>
    from .config import (
ModuleNotFoundError: No module named 'model.config'
Traceback (most recent call last):
  File "/kaggle/working/AethyxLM/train.py", line 22, in <module>
    from model.gpt import GPT
  File "/kaggle/working/AethyxLM/model/gpt.py", line 8, in <module>
    from

In [70]:
# ============================================================
# CELL 9: DOWNLOAD CHECKPOINTS & LOGS
# ============================================================
from IPython.display import FileLink, display
import os

print("Checkpoints & Log Files available for download:")
ckpt_list = []
if os.path.exists('checkpoints'):
    for f in sorted(os.listdir('checkpoints')):
        if f.endswith('.pt'):
            ckpt_list.append(os.path.join('checkpoints', f))

for f in ckpt_list:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f'File: {f} ({size_mb:.1f} MB)')
    display(FileLink(f))

if os.path.exists('logs'):
    for root, _, files in os.walk('logs'):
        for f in files:
            path = os.path.join(root, f)
            print(f'Log: {path}')
            display(FileLink(path))

Checkpoints & Log Files available for download:
Log: logs/run_config.json


/kaggle/working/AethyxLM/logs/run_config.json

In [71]:
# ============================================================
# CELL 10: INFERENCE & TEXT GENERATION TEST
# ============================================================
import os, torch
from model.gpt import GPT
from tokenizer.tokenizer import AethyxTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'

ckpt_path = 'checkpoints/checkpoint_best.pt'
if not os.path.exists(ckpt_path):
    ckpt_path = 'checkpoints/checkpoint_latest.pt'

if not os.path.exists(ckpt_path):
    print("No checkpoint found for generation test.")
else:
    print(f"Loading checkpoint for generation: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    
    model = GPT().to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    
    tok = AethyxTokenizer()
    
    @torch.no_grad()
    def generate(prompt, max_new=150, temp=0.8, top_k=50):
        ids = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)
        for _ in range(max_new):
            logits = model(ids[:, -128:])
            logits = logits[:, -1, :] / temp
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            ids = torch.cat([ids, next_id], dim=1)
        return tok.decode(ids[0].tolist())

    prompts = [
        "Once upon a time",
        "The little girl",
        "In a small kingdom"
    ]
    
    for p in prompts:
        print(f"\nPrompt: {p}")
        print("-" * 40)
        output = generate(p, max_new=100)
        print(output)
        print("=" * 60)

No checkpoint found for generation test.
